In [0]:
%sql
SELECT * FROM ECOMMERCE.BRONZE.PRODUCTS LIMIT 10;


## Loading libraries for future usage

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col


## Loading products data 

In [0]:
products_df = (
    spark.read.table("ecommerce.bronze.products")
    .withColumn("updated_at", col("updated_at").cast("timestamp"))
    .withColumn("price", col("price").cast("decimal(18,2)"))
    .withColumn("cost", col("cost").cast("decimal(18,2)"))
    .dropDuplicates()
    .filter(col("price") > 0)
    .filter(col("cost") > 0)
)

## Business logic 1: Retain latest product information

In [0]:
window_spec = Window.partitionBy("product_id").orderBy(col("updated_at").desc())
products_df = products_df.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1).drop("rn")

## Standardize product names and statuses

In [0]:
products_df = products_df.withColumn(
    "product_name", initcap(trim(col("product_name")))
).withColumn("product_status", upper(trim(col("product_status"))))

In [0]:
products_df.write.format("delta").mode("append").saveAsTable(
    "ecommerce.silver.products"
)

In [0]:
%sql
SELECT * FROM ECOMMERCE.SILVER.PRODUCTS LIMIT 10;